In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 150)

In [2]:
PROJECT_ROOT = Path("E:\\Data Analysis Projects\\Product & Revenue Intelligence Platform\\datasets")

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

In [3]:
## Read all darasets
datasets = {
    "customers": pd.read_csv(RAW_DATA_PATH / "olist_customers_dataset.csv"),
    "geolocation": pd.read_csv(RAW_DATA_PATH / "olist_geolocation_dataset.csv"),
    "orders": pd.read_csv(RAW_DATA_PATH / "olist_orders_dataset.csv"),
    "order_items": pd.read_csv(RAW_DATA_PATH / "olist_order_items_dataset.csv"),
    "payments": pd.read_csv(RAW_DATA_PATH / "olist_order_payments_dataset.csv"),
    "reviews": pd.read_csv(RAW_DATA_PATH / "olist_order_reviews_dataset.csv"),
    "products": pd.read_csv(RAW_DATA_PATH / "olist_products_dataset.csv"),
    "sellers": pd.read_csv(RAW_DATA_PATH / "olist_sellers_dataset.csv"),
    "category_translation": pd.read_csv(
        RAW_DATA_PATH / "product_category_name_translation.csv"
    )
}

In [4]:
## Data Inventory
inventory = []

for table_name, df in datasets.items():

    inventory.append({
        "table": table_name,
        "rows": len(df),
        "columns": df.shape[1],
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2)
    })

inventory_df = pd.DataFrame(inventory)

inventory_df

,table,rows,columns,memory_mb
0,customers,99441,5,26.59
1,geolocation,1000163,5,129.38
2,orders,99441,8,52.94
3,order_items,112650,7,35.99
4,payments,103886,5,16.23
5,reviews,99224,7,39.12
6,products,32951,9,6.30
7,sellers,3095,4,0.59
8,category_translation,71,2,0.01


In [8]:
## save inventory
inventory_df.to_csv(
    PROCESSED_DATA_PATH / "inventory.csv",
    index=False
)

In [9]:
## column metadata function
def build_column_metadata(df, table_name):

    metadata = pd.DataFrame({
        "table": table_name,
        "column": df.columns,
        "data_type": df.dtypes.astype(str).values,
        "non_null_count": df.notna().sum().values,
        "null_count": df.isna().sum().values,
        "null_percent": (
            df.isna().mean() * 100
        ).round(2).values,
        "unique_values": df.nunique(dropna=True).values
    })

    return metadata

In [10]:
## built metadata
metadata_frames = []

for table_name, df in datasets.items():

    metadata_frames.append(
        build_column_metadata(df, table_name)
    )

column_metadata = pd.concat(
    metadata_frames,
    ignore_index=True
)

column_metadata.head()

,table,column,data_type,non_null_count,null_count,null_percent,unique_values
0,customers,customer_id,object,99441,0,0.0,99441
1,customers,customer_unique_id,object,99441,0,0.0,96096
2,customers,customer_zip_code_prefix,int64,99441,0,0.0,14994
3,customers,customer_city,object,99441,0,0.0,4119
4,customers,customer_state,object,99441,0,0.0,27


In [11]:
## save metadata
column_metadata.to_csv(
    PROCESSED_DATA_PATH / "column_metadata.csv",
    index=False
)

In [13]:
## Duplicate Report
duplicate_report = []

for table_name, df in datasets.items():

    duplicate_report.append({

        "table": table_name,

        "rows": len(df),

        "duplicate_rows": df.duplicated().sum(),

        "duplicate_percent":
            round(df.duplicated().mean()*100,4)

    })

duplicate_report = pd.DataFrame(
    duplicate_report
)

duplicate_report

,table,rows,duplicate_rows,duplicate_percent
0,customers,99441,0,0.0000
1,geolocation,1000163,261831,26.1788
2,orders,99441,0,0.0000
3,order_items,112650,0,0.0000
4,payments,103886,0,0.0000
5,reviews,99224,0,0.0000
6,products,32951,0,0.0000
7,sellers,3095,0,0.0000
8,category_translation,71,0,0.0000


In [14]:
duplicate_report.to_csv(
    PROCESSED_DATA_PATH / "duplicate_report.csv",
    index=False
)

In [15]:
geolocation = datasets["geolocation"]

print("Total Rows:")
print(len(geolocation))

print()

print("Unique ZIP Codes:")
print(geolocation["geolocation_zip_code_prefix"].nunique())

print()

print("Duplicate ZIP Codes:")
print(
    geolocation["geolocation_zip_code_prefix"].duplicated().sum()
)

Total Rows:
1000163

Unique ZIP Codes:
19015

Duplicate ZIP Codes:
981148


In [16]:
geolocation[
    geolocation.duplicated()
].head(10)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
15,1046,-23.546081,-46.644820,sao paulo,SP
44,1046,-23.546081,-46.644820,sao paulo,SP
65,1046,-23.546081,-46.644820,sao paulo,SP
66,1009,-23.546935,-46.636588,sao paulo,SP
67,1046,-23.546081,-46.644820,sao paulo,SP
72,1046,-23.545320,-46.644069,sao paulo,SP
79,1050,-23.549854,-46.643139,sao paulo,SP
80,1032,-23.540775,-46.635515,sao paulo,SP
82,1046,-23.546081,-46.644820,sao paulo,SP
86,1048,-23.547449,-46.640169,são paulo,SP


In [17]:
def candidate_keys(df):

    report = []

    total_rows = len(df)

    for column in df.columns:

        report.append({

            "column": column,

            "unique_values": df[column].nunique(dropna=False),

            "null_count": df[column].isna().sum(),

            "is_unique":
                df[column].is_unique,

            "candidate_pk":
                df[column].is_unique and df[column].isna().sum()==0

        })

    return pd.DataFrame(report)

In [18]:
for table_name, df in datasets.items():

    print("="*70)

    print(table_name.upper())

    display(candidate_keys(df))

CUSTOMERS


,column,unique_values,null_count,is_unique,candidate_pk
0,customer_id,99441,0,True,True
1,customer_unique_id,96096,0,False,False
2,customer_zip_code_prefix,14994,0,False,False
3,customer_city,4119,0,False,False
4,customer_state,27,0,False,False


GEOLOCATION


,column,unique_values,null_count,is_unique,candidate_pk
0,geolocation_zip_code_prefix,19015,0,False,False
1,geolocation_lat,717360,0,False,False
2,geolocation_lng,717613,0,False,False
3,geolocation_city,8011,0,False,False
4,geolocation_state,27,0,False,False


ORDERS


...

,column,unique_values,null_count,is_unique,candidate_pk
0,product_id,32951,0,True,True
1,product_category_name,74,610,False,False
2,product_name_lenght,67,610,False,False
3,product_description_lenght,2961,610,False,False
4,product_photos_qty,20,610,False,False
5,product_weight_g,2205,2,False,False
6,product_length_cm,100,2,False,False
7,product_height_cm,103,2,False,False
8,product_width_cm,96,2,False,False


SELLERS


,column,unique_values,null_count,is_unique,candidate_pk
0,seller_id,3095,0,True,True
1,seller_zip_code_prefix,2246,0,False,False
2,seller_city,611,0,False,False
3,seller_state,23,0,False,False


CATEGORY_TRANSLATION


,column,unique_values,null_count,is_unique,candidate_pk
0,product_category_name,71,0,True,True
1,product_category_name_english,71,0,True,True


In [19]:
order_items = datasets["order_items"]

order_items[
    ["order_id", "order_item_id"]
].duplicated().sum()

0

In [20]:
payments = datasets["payments"]

payments[
    ["order_id", "payment_sequential"]
].duplicated().sum()

0

In [21]:
reviews = datasets["reviews"]

reviews[
    ["review_id", "order_id"]
].duplicated().sum()

0

In [22]:
reviews["review_id"].value_counts().head(20)

review_id
7b606b0d57b078384f0b58eac1d41d78    3
dbdf1ea31790c8ecfcc6750525661a9b    3
32415bbf6e341d5d517080a796f79b5c    3
0c76e7a547a531e7bf9f0b99cba071c1    3
4219a80ab469e3fc9901437b73da3f75    3
abbfacb2964f74f6487c9c10ac46daa6    3
e44840754f12fad2b8646712121b349a    3
70509c441d994fa03d6c1457930c9024    3
2172867fd5b1a55f98fe4608e1547b4b    3
832acec9bbf4efe65c3fb6423d8b4ed7    3
2d6ac45f859465b5c185274a1c929637    3
4d0e6dd087008d1f992d25ef6e1f619f    3
4548534449b1f572e357211b90724f1b    3
f4bb9d6dd4fb6dcc2298f0e7b17b8e1e    3
44e9f871226d8a130de3fc39dfbdf0c5    3
1fb4ddc969e6bea80e38deec00393a6f    3
39b4603793c1c7f5f36d809b4a218664    3
08528f70f579f0c830189efc523d2182    3
38821b5c496b678cf91acc34892805ad    3
9e25d6e3025e9b9a0fc7f03588d33e2b    3
Name: count, dtype: int64

# 6. Enterprise Data Model

## Objective

Identify business entities, validate relationships, and define the analytical data model before designing the data warehouse.

In [5]:
## hellper function
def relationship_validation(parent_df, child_df, parent_key, child_key):
    parent_values = set(parent_df[parent_key].dropna().unique())
    child_values = set(child_df[child_key].dropna().unique())

    orphan_keys = child_values - parent_values

    return {
        "parent_table": parent_key,
        "child_table": child_key,
        "parent_unique": parent_df[parent_key].nunique(),
        "child_unique": child_df[child_key].nunique(),
        "orphan_keys": len(orphan_keys),
        "referential_integrity": len(orphan_keys) == 0
    }

In [6]:
relationship_report = []

relationship_report.append(
    relationship_validation(
        datasets["customers"],
        datasets["orders"],
        "customer_id",
        "customer_id"
    )
)

relationship_report.append(
    relationship_validation(
        datasets["orders"],
        datasets["order_items"],
        "order_id",
        "order_id"
    )
)

relationship_report.append(
    relationship_validation(
        datasets["products"],
        datasets["order_items"],
        "product_id",
        "product_id"
    )
)

relationship_report.append(
    relationship_validation(
        datasets["sellers"],
        datasets["order_items"],
        "seller_id",
        "seller_id"
    )
)

relationship_report.append(
    relationship_validation(
        datasets["orders"],
        datasets["payments"],
        "order_id",
        "order_id"
    )
)

relationship_report.append(
    relationship_validation(
        datasets["orders"],
        datasets["reviews"],
        "order_id",
        "order_id"
    )
)

relationship_report = pd.DataFrame(relationship_report)

relationship_report

,parent_table,child_table,parent_unique,child_unique,orphan_keys,referential_integrity
0,customer_id,customer_id,99441,99441,0,True
1,order_id,order_id,99441,98666,0,True
2,product_id,product_id,32951,32951,0,True
3,seller_id,seller_id,3095,3095,0,True
4,order_id,order_id,99441,99440,0,True
5,order_id,order_id,99441,98673,0,True


In [7]:
relationship_report.to_csv(
    PROCESSED_DATA_PATH / "relationship_report.csv",
    index=False
)

In [8]:
orders_per_customer = (
    datasets["orders"]
    .groupby("customer_id")
    .size()
    .reset_index(name="order_count")
)

orders_per_customer["order_count"].describe()

count    99441.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
Name: order_count, dtype: float64

In [9]:
orders_per_customer["order_count"].value_counts().sort_index()

order_count
1    99441
Name: count, dtype: int64

# 8. Data Type Profiling

## Objective

Determine optimal SQL data types for each source table before creating the raw database schema.

In [10]:
def profile_string_lengths(df):
    report = []

    for col in df.select_dtypes(include="object").columns:

        max_len = df[col].fillna("").astype(str).str.len().max()

        report.append({
            "column": col,
            "max_length": max_len
        })

    return pd.DataFrame(report)

In [11]:
for name, df in datasets.items():

    print("=" * 70)
    print(name.upper())

    display(profile_string_lengths(df))

CUSTOMERS


,column,max_length
0,customer_id,32
1,customer_unique_id,32
2,customer_city,32
3,customer_state,2


GEOLOCATION


,column,max_length
0,geolocation_city,38
1,geolocation_state,2


ORDERS


...

,column,max_length
0,product_id,32
1,product_category_name,46


SELLERS


,column,max_length
0,seller_id,32
1,seller_city,40
2,seller_state,2


CATEGORY_TRANSLATION


,column,max_length
0,product_category_name,46
1,product_category_name_english,39


In [12]:
dtype_profile = {}

for name, df in datasets.items():

    dtype_profile[name] = profile_string_lengths(df)

dtype_profile

{'customers':                column  max_length
 0         customer_id          32
 1  customer_unique_id          32
 2       customer_city          32
 3      customer_state           2,
 'geolocation':               column  max_length
 0   geolocation_city          38
 1  geolocation_state           2,
 'orders':                           column  max_length
 0                       order_id          32
 1                    customer_id          32
 2                   order_status          11
 3       order_purchase_timestamp          19
 4              order_approved_at          19
 5   order_delivered_carrier_date          19
 6  order_delivered_customer_date          19
 7  order_estimated_delivery_date          19,
 'order_items':                 column  max_length
 0             order_id          32
 1           product_id          32
 2            seller_id          32
 3  shipping_limit_date          19,
 'payments':          column  max_length
 0      order_id          32
 1

In [13]:
numeric_profile = []

for table_name, df in datasets.items():

    numeric_cols = df.select_dtypes(include=["number"]).columns

    for col in numeric_cols:

        numeric_profile.append({

            "table": table_name,

            "column": col,

            "dtype": str(df[col].dtype),

            "min": df[col].min(),

            "max": df[col].max(),

            "nulls": df[col].isna().sum()

        })

numeric_profile = pd.DataFrame(numeric_profile)

numeric_profile

,table,column,dtype,min,max,nulls
0,customers,customer_zip_code_prefix,int64,1003.000000,99990.000000,0
1,geolocation,geolocation_zip_code_prefix,int64,1001.000000,99990.000000,0
2,geolocation,geolocation_lat,float64,-36.605374,45.065933,0
3,geolocation,geolocation_lng,float64,-101.466766,121.105394,0
4,order_items,order_item_id,int64,1.000000,21.000000,0
5,order_items,price,float64,0.850000,6735.000000,0
6,order_items,freight_value,float64,0.000000,409.680000,0
7,payments,payment_sequential,int64,1.000000,29.000000,0
8,payments,payment_installments,int64,0.000000,24.000000,0
9,payments,payment_value,float64,0.000000,13664.080000,0


In [14]:
numeric_profile.to_csv(
    PROCESSED_DATA_PATH / "numeric_profile.csv",
    index=False
)

In [15]:
datetime_profile = []

for table_name, df in datasets.items():

    for col in df.columns:

        if "date" in col.lower() or "timestamp" in col.lower():

            datetime_profile.append({

                "table": table_name,

                "column": col,

                "sample": df[col].dropna().iloc[0]

            })

pd.DataFrame(datetime_profile)

,table,column,sample
0,orders,order_purchase_timestamp,2017-10-02 10:56:33
1,orders,order_delivered_carrier_date,2017-10-04 19:55:00
2,orders,order_delivered_customer_date,2017-10-10 21:25:13
3,orders,order_estimated_delivery_date,2017-10-18 00:00:00
4,order_items,shipping_limit_date,2017-09-19 09:45:35
5,reviews,review_creation_date,2018-01-18 00:00:00
6,reviews,review_answer_timestamp,2018-01-18 21:46:59
